In [37]:
import pandas as pd
import argparse
import logging
import sys
import os


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def create_mock_args():
    class MockArgs:
        def __init__(self):
            self.input_csv = 'sample_data.csv'
            self.output_excel = 'output.xlsx'
            self.na_fill_value = 'NA'
            self.date_cols = 'Order Date'
            self.rename_cols = 'Product ID:ProductID_New,Product Name:ProductName_New' # Default rename map for Colab

    logging.info("Using mock arguments for Colab environment.")
    return MockArgs()


if 'google.colab' in sys.modules:
    args = create_mock_args()
else:
    parser = argparse.ArgumentParser(description='Convert CSV to Excel with data cleaning.')
    parser.add_argument('--input_csv', type=str, required=True, help='Path to the input CSV file.')
    parser.add_argument('--output_excel', type=str, required=True, help='Path to the output Excel file.')
    parser.add_argument('--na_fill_value', type=str, default='NA', help='Value to fill missing (NaN) entries.')
    parser.add_argument('--date_cols', type=str, default='', help='Comma-separated list of columns to convert to datetime.')
    parser.add_argument('--rename_cols', type=str, default='', help='Comma-separated list of old_name:new_name pairs for column renaming.')

    try:
        args = parser.parse_args()
    except SystemExit as e:
        logging.error(f"Argparse exited: {e}. Falling back to mock arguments for interactive execution.")
        args = create_mock_args()

logging.info(f"Parsed arguments: {args.__dict__}")


dummy_data = {
    'Product ID': [101, 102, None, 104],
    'Product Name': ['Laptop', 'Mouse', 'Keyboard', 'Monitor'],
    'Order Date': ['2023-01-01', '2023-01-05', 'invalid-date', '2023-01-10'],
    'Price': [1200, 25, 75, None]
}
dummy_df = pd.DataFrame(dummy_data)
dummy_df.to_csv('sample_data.csv', index=False)
logging.info("Created 'sample_data.csv' for demonstration purposes.")

In [38]:
def csv_to_excel_converter(input_csv, output_excel, na_fill_value, date_cols, rename_cols):
    logging.info(f"Starting conversion for {input_csv} to {output_excel}")
    try:
        # Read the CSV file
        df = pd.read_csv(input_csv)
        logging.info(f"Successfully read {input_csv}. Shape: {df.shape}")

        # Fill missing values
        # Iterate through columns to handle mixed types when filling with a string
        for col in df.columns:
            if df[col].isnull().any(): # Check if there are any NaNs in the column
                # If the column is numeric and we are filling with a string, convert existing numbers to string first
                if pd.api.types.is_numeric_dtype(df[col]) and isinstance(na_fill_value, str):
                    df[col] = df[col].apply(lambda x: str(x) if pd.notna(x) else x)
                    df[col] = df[col].fillna(na_fill_value) # Use df[col] = to ensure assignment
                else:
                    df[col].fillna(na_fill_value, inplace=True)
        logging.info(f"Filled missing values with '{na_fill_value}'.")

        # Convert specified columns to datetime
        if date_cols:
            for col in date_cols.split(','):
                col = col.strip()
                if col in df.columns:
                    original_dtype = df[col].dtype
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                    logging.info(f"Converted column '{col}' to datetime. Original dtype: {original_dtype}, New dtype: {df[col].dtype}")
                else:
                    logging.warning(f"Date column '{col}' not found in the DataFrame. Skipping.")

        # Rename columns
        if rename_cols:
            rename_map = {}
            for pair in rename_cols.split(','):
                if ':' in pair:
                    old_name, new_name = pair.split(':', 1)
                    old_name, new_name = old_name.strip(), new_name.strip()
                    if old_name in df.columns:
                        rename_map[old_name] = new_name
                    else:
                        logging.warning(f"Column '{old_name}' to be renamed not found in DataFrame. Skipping.")
                else:
                    logging.warning(f"Invalid rename format for '{pair}'. Expected 'old_name:new_name'. Skipping.")
            if rename_map:
                df.rename(columns=rename_map, inplace=True)
                logging.info(f"Renamed columns: {rename_map}")

        # Export to Excel
        df.to_excel(output_excel, index=False, engine='openpyxl')
        logging.info(f"Successfully converted and saved to {output_excel}")

    except FileNotFoundError:
        logging.error(f"Input CSV file not found at: {input_csv}")
        print(f"Error: Input CSV file not found at {input_csv}")
    except Exception as e:
        logging.error(f"An error occurred during conversion: {e}", exc_info=True)
        print(f"An unexpected error occurred: {e}")

In [39]:
print("Dummy DataFrame before cleaning:")
display(dummy_df)

Dummy DataFrame before cleaning:


,Product ID,Product Name,Order Date,Price
0,101.0,Laptop,2023-01-01,1200.0
1,102.0,Mouse,2023-01-05,25.0
2,NaN,Keyboard,invalid-date,75.0
3,104.0,Monitor,2023-01-10,NaN


In [40]:

dummy_df_cleaned = dummy_df.fillna('NA')

dummy_df_cleaned['Order Date'] = pd.to_datetime(dummy_df_cleaned['Order Date'], errors='coerce')

print("Dummy DataFrame after cleaning:")
display(dummy_df_cleaned)

Dummy DataFrame after cleaning:


,Product ID,Product Name,Order Date,Price
0,101.0,Laptop,2023-01-01,1200.0
1,102.0,Mouse,2023-01-05,25.0
2,NA,Keyboard,NaT,75.0
3,104.0,Monitor,2023-01-10,NA
